# Notebook 5: Knowledge discovery through facets

This notebook demonstrates how facet relations expose useful bridges inside the knowledge graph: similar claims become connected through shared cores, scoped differences, and timeframe differences that support discovery.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.active_memory.models import WorkingItem
from src.manifold_sidecar import ManifoldRankingService
from src.persistence.pipeline import MutationPipeline
from src.retrieval.composer import RetrievalComposer
from src.terminus.adapter import TerminusMemoryRepository

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-05"
shutil.rmtree(memory_root, ignore_errors=True)
repo = TerminusMemoryRepository(url="http://localhost:9999")
service = ManifoldRankingService(model_id="facet-discovery-v1", geometry_version="facet-geometry-v1")
pipeline = MutationPipeline(
    memory_root,
    enable_terminus=True,
    terminus_repo=repo,
    manifold_service=service,
    enable_inference=True,
)
composer = RetrievalComposer(memory_root, terminus_repo=repo)

In [ ]:
result = pipeline.run(
    WorkingItem(
        item_type="observation",
        content=(
            "Yesterday, Acme Payments delayed refund processing for enterprise customers in Europe. "
            "Today, Acme Payments restored refund processing for enterprise customers worldwide after the retry queue cleared."
        ),
        session_id="facet-discovery",
    ),
    session_id="facet-discovery",
)
show(result.__dict__)

In [ ]:
facet_relations = repo.query_facet_relations(result.inference_branch)
inference_nodes = repo.query_inference_nodes(result.inference_branch)
show({
    "inference_branch": result.inference_branch,
    "facet_relations": facet_relations,
    "inference_nodes": inference_nodes,
})

In [ ]:
exploratory = composer.retrieve(
    include_terminus=True,
    include_speculative=True,
    inference_branch=result.inference_branch,
)
facet_index = {}
for relation in exploratory["facet_relations"]:
    facet_index.setdefault(relation["shared_core_claim"], []).append({
        "facet_type": relation["facet_type"],
        "differences": relation["differences"],
        "facet_strength": relation["facet_strength"],
        "relatedness_score": relation["relatedness_score"],
    })
show({
    "discovery_context": exploratory,
    "facet_index": facet_index,
})

## What this notebook demonstrates

- speculative graph generation from existing memory claims
- facets as relations rather than standalone concept nodes
- discovery of adjacent knowledge through shared core claims and explicit differences
- ranked facet metadata that helps decide which bridges are worth exploring first